# Part 3: InBody Report Image → Structured Data Conversion (via OpenRouter)

---

이 Task에서는 비정형 데이터인 **InBody Report 이미지**를 분석하여, 필요한 측정값을 **구조화된 Excel 데이터** 형태로 자동 정리하는 작업을 수행합니다.

### Background

인바디 리포트는 체성분(체중, 체지방률, 골격근량 등), 기초대사량, 신체 균형 지표 등 다양한 정보를 **이미지 형태**로 포함하고 있어, 직접 수기로 정리하기 어렵고 시간이 많이 소요됩니다.

### Objective

GPT Vision 모델을 활용하여 인바디 이미지에서 핵심 수치·측정값을 **자동으로 추출**하고, 이를 **엑셀(Excel) 형태의 구조화된 표(tabular data)** 로 변환합니다.

### Why OpenRouter?

- **Zero Data Retention (ZDR)**: 프롬프트 및 응답 데이터가 제공자 서버에 저장되지 않습니다.
- **`data_collection: deny`**: 제공자의 모델 학습에 데이터가 사용되지 않습니다.
- 의료 데이터(InBody 리포트)를 다룰 때 **프라이버시 보호**가 특히 중요합니다.

| Step | Description |
|------|-------------|
| **3.1** | 이미지 로드 및 Base64 인코딩 |
| **3.2** | GPT Vision으로 데이터 추출 (ZDR 적용) |
| **3.3** | 텍스트 → DataFrame 변환 |
| **3.4** | CSV 저장 |

### Requirements

- `openai` >= 1.0
- OpenRouter API Key ([https://openrouter.ai/keys](https://openrouter.ai/keys))

---
## 0. Environment Setup

In [1]:
from openai import OpenAI
import pandas as pd
import base64
import os
import re

# ── Configuration ──────────────────────────────────────────────
PATH = "./DATA/"
OPENROUTER_API_KEY = ""  # "" 안에 openrouter api key를 복사해서 붙여넣기를 진행합니다.
MODEL = "openai/gpt-5.6"

In [ ]:
# ── Download sample InBody report image ───────────────────────
BASE_URL = "https://raw.githubusercontent.com/leelabsg/clinical-llm-tutorial/main/Session2"
!wget -qP $PATH {BASE_URL}/inbody_sample.jpg

print("Download complete.")

---
## 1. Initialize OpenRouter Client

OpenAI Python SDK의 `base_url`을 OpenRouter로 지정하여 동일한 인터페이스로 사용합니다.

**Privacy 설정:**
- `zdr: True` — Zero Data Retention: 제공자가 데이터를 저장하지 않는 엔드포인트로만 라우팅
- `data_collection: "deny"` — 프롬프트/응답이 모델 학습에 사용되지 않도록 차단

In [3]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# ── Privacy provider config (reused across all requests) ──────
PROVIDER_CONFIG = {
    "zdr": True,
    "data_collection": "deny",
}

---
## 3.1 이미지 로드 및 인코딩

GPT Vision API에 이미지를 전달하려면 **Base64 인코딩**이 필요합니다.

In [4]:
# 이미지 파일 로드 및 base64 인코딩
image_path = os.path.join(PATH, "inbody_sample.jpg")

with open(image_path, "rb") as f:
    img_bytes = f.read()
    img_base64 = base64.b64encode(img_bytes).decode("utf-8")

print(f"이미지 로드 완료: {image_path}")
print(f"Base64 인코딩 길이: {len(img_base64):,} chars")

이미지 로드 완료: /Users/moonie/Desktop/2026_LeeLab/KOGO_OpenAI_API_Tutorial_2025/Session2/inbody_sample.jpg
Base64 인코딩 길이: 484,336 chars


---
## 3.2 GPT Vision으로 데이터 추출 (ZDR 적용)

GPT Vision 모델에 InBody 리포트 이미지를 전달하고, 구조화된 형식으로 측정값을 추출합니다.  
모든 요청에 `zdr: True`, `data_collection: "deny"`가 적용됩니다.

In [5]:
# InBody 데이터 추출 프롬프트
prompt_inbody = """
Extract the following values from the InBody report image.
Return in the exact format below. If not visible, leave blank.

## Basic Information
1. ID:
2. Height:
3. Age:
4. Gender:
5. Test Date/Time:

## Key Measurements
1. Weight:
2. Skeletal Muscle Mass (SMM):
3. Body Fat Mass:
4. BMI:
5. Percent Body Fat (PBF):
"""

In [6]:
# Vision 모델로 InBody 이미지 분석 (ZDR 적용)
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are an AI assistant that extracts structured measurement values from an InBody body composition analysis report image.",
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt_inbody},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{img_base64}"
                    },
                },
            ],
        },
    ],
    extra_body={"provider": PROVIDER_CONFIG},
)

inbody_text = response.choices[0].message.content
print(inbody_text)

## Basic Information
1. ID: Jane Doe
2. Height: 163 cm
3. Age: 41
4. Gender: Female
5. Test Date/Time: 2017.03.08. 16:47

## Key Measurements
1. Weight: 66.4 kg
2. Skeletal Muscle Mass (SMM): 26.7 kg
3. Body Fat Mass: 18.1 kg
4. BMI: 25.0 kg/m²
5. Percent Body Fat (PBF): 27.2%


---
## 3.3 텍스트 → DataFrame 변환

GPT가 반환한 텍스트를 파싱하여 `pandas DataFrame`으로 변환합니다.

In [7]:
def parse_inbody_text_to_row(text):
    """
    InBody 텍스트 블록을 받아서 {'항목명': 값} 딕셔너리로 파싱
    """
    row_data = {}
    lines = text.strip().split("\n")

    for line in lines:
        line = line.strip()

        # section header는 건너뛰기
        if line.startswith("##"):
            continue

        # key:value 패턴 매칭
        # (?:\d+\.\s*)? = "1. " 같은 번호가 있어도 무시
        # (.+?)          = : 앞에 오는 문자열을 key로
        # \s*(.*)        = : 뒤에 오는 내용을 value로
        match = re.match(r"(?:\d+\.\s*)?(.+?):\s*(.*)", line)

        if match:
            key = match.group(1).strip()
            value = match.group(2).strip()
            row_data[key] = value

    return row_data

In [8]:
# 파싱 및 DataFrame 생성
parsed_row = parse_inbody_text_to_row(inbody_text)
df = pd.DataFrame([parsed_row])
df.head()

,ID,Height,Age,Gender,Test Date/Time,Weight,Skeletal Muscle Mass (SMM),Body Fat Mass,BMI,Percent Body Fat (PBF)
0,Jane Doe,163 cm,41,Female,2017.03.08. 16:47,66.4 kg,26.7 kg,18.1 kg,25.0 kg/m²,27.2%


---
## 3.4 CSV 저장

추출된 데이터를 CSV 파일로 저장합니다.

In [ ]:
# CSV 파일로 저장
csv_path = os.path.join(PATH, "inbody_sample.csv")
df.to_csv(csv_path, index=False)
print(f"저장 완료: {csv_path}")

---
## Usage Check

In [9]:
print(response.usage)

CompletionUsage(completion_tokens=124, prompt_tokens=1042, total_tokens=1166, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0, cache_write_tokens=1039, video_tokens=0), cost=0.01022875, is_byok=False, cost_details={'upstream_inference_cost': 0.01022875, 'upstream_inference_prompt_cost': 0.00650875, 'upstream_inference_completions_cost': 0.00372})


---

## Summary

| Step | Input | Output |
|------|-------|--------|
| **이미지 로드** | `inbody_sample.jpg` | Base64 encoded string |
| **GPT Vision 추출** | Base64 image + prompt | 구조화된 텍스트 (key: value) |
| **DataFrame 변환** | 텍스트 파싱 | `pandas.DataFrame` |
| **CSV 저장** | DataFrame | `inbody_sample.csv` |

### OpenRouter Privacy Settings

| Parameter | Value | Effect |
|-----------|-------|--------|
| `zdr` | `True` | ZDR 정책을 준수하는 엔드포인트로만 라우팅 |
| `data_collection` | `"deny"` | 프롬프트/응답이 모델 학습에 사용되지 않음 |

### Key Takeaways

- GPT Vision은 이미지 속 텍스트와 수치를 높은 정확도로 추출할 수 있습니다.
- Prompt에 **정확한 출력 형식**을 지정하면 후처리(파싱)가 용이합니다.
- 다수의 InBody 이미지에 동일한 파이프라인을 적용하면 **대량 데이터 구조화**가 가능합니다.
- OpenRouter의 ZDR 설정을 통해 **의료 데이터 프라이버시**를 보호할 수 있습니다.